# ChuckleNet GPU Validation
## AST Label Validation + CNN Training on 481 Videos

**Goal:**
1. Run pre-trained AST model on GPU to independently validate energy-based labels
2. Train CNN on mel spectrograms (end-to-end, no hand-crafted features)
3. Compare AST/CNN/Energy approaches

**Runtime:** GPU (T4)

In [ ]:
# Step 1: Setup
!pip install -q transformers librosa kaggle
import torch, numpy as np, librosa, json, os, time
from pathlib import Path
from collections import defaultdict, Counter
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Step 2: Download data from Kaggle
!mkdir -p ~/.kaggle && cp /content/drive/MyDrive/.kaggle/kaggle.json ~/.kaggle/ 2>/dev/null || true
!kaggle datasets download -d subhajitdas/chuckle-wavlm-555-videos --unzip -q -p /tmp/wavlm_data

# Upload energy labels from local
from google.colab import files
print('Upload energy_labels_668.npz and mfcc_668.npz from local:')
# These need to be uploaded manually or from Drive
# For now, we'll extract features fresh on GPU

In [ ]:
# Step 3: Mount Drive for audio + utterances
from google.colab import drive
drive.mount('/content/gdrive')

BASE = Path('/content/gdrive/MyDrive')
UTT_PATH = BASE / 'utterances_clean.jsonl'

# Find utterances
if UTT_PATH.exists():
    with open(UTT_PATH) as f:
        utts = [json.loads(l) for l in f]
    print(f'Utterances: {len(utts):,}')
else:
    print('utterances_clean.jsonl not found. Please upload.')
    utts = []

utts_by_vid = defaultdict(list)
for u in utts:
    utts_by_vid[u['video_id']].append(u)
print(f'Videos: {len(utts_by_vid)}')

In [ ]:
# Step 4: Find audio files
SR = 16000
audio_dirs = [
    BASE / 'chuckle_audio',
    BASE / 'chuckle_audio_all' / 'audio',
    BASE / 'chuckle_audio_all' / 'audio_final',
    BASE / 'chuckle_audio_all' / 'audio_new',
    BASE / 'chuckle_audio_all' / 'audio_all',
]

audio_map = {}
for d in audio_dirs:
    if d.exists():
        for p in d.iterdir():
            if p.suffix in ['.mp3', '.wav', '.m4a'] and not p.name.endswith('.part'):
                vid = p.stem
                if vid not in audio_map:
                    audio_map[vid] = str(p)

print(f'Audio files found: {len(audio_map)}')
print(f'Videos with utterances: {len(utts_by_vid)}')
overlap = set(audio_map.keys()) & set(utts_by_vid.keys())
print(f'Overlap: {len(overlap)}')

## Part A: AST Label Validation

In [ ]:
# Step 5: Load AST model (runs on GPU!)
from transformers import ASTForAudioClassification, AutoFeatureExtractor

print('Loading AST model on GPU...')
ast_model = ASTForAudioClassification.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593').to(device)
ast_extractor = AutoFeatureExtractor.from_pretrained('MIT/ast-finetuned-audioset-10-10-0.4593')
ast_model.eval()
laughter_id = 16  # 'Laughter'
print('AST model loaded!')

In [ ]:
# Step 6: Run AST on ALL videos (GPU = fast!)
ast_labels = {}  # vid -> [(start, end, ast_prob)]
t0 = time.time()

videos = sorted(overlap)
for vi, vid in enumerate(videos):
    audio_path = audio_map[vid]
    try:
        y_full, sr = librosa.load(audio_path, sr=SR, mono=True)
    except:
        continue
    
    vid_labels = []
    vid_utts = utts_by_vid[vid]
    
    # Process every 3rd utterance (for speed)
    for u in vid_utts[::3]:
        start = int(u['start'] * SR)
        end = int(u['end'] * SR)
        if end > len(y_full): end = len(y_full)
        seg = y_full[start:end]
        if len(seg) < SR * 0.1: continue
        
        # Pad to 10 seconds
        if len(seg) < 160000:
            seg = np.pad(seg, (0, 160000 - len(seg)))
        else:
            seg = seg[:160000]
        
        inputs = ast_extractor(seg, sampling_rate=SR, return_tensors='pt')
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            probs = torch.softmax(ast_model(**inputs).logits, dim=-1)
            laugh_prob = probs[0, laughter_id].item()
        
        # Compute rel_energy
        full_rms = np.sqrt(np.mean(y_full**2)) + 1e-8
        seg_rms = np.sqrt(np.mean(seg**2)) if len(seg) > 0 else 0
        rel_e = seg_rms / full_rms
        
        vid_labels.append({
            'start': u['start'], 'end': u['end'],
            'ast_prob': laugh_prob,
            'rel_energy': rel_e,
            'vtt_label': u.get('label', 0),
        })
    
    ast_labels[vid] = vid_labels
    
    if (vi+1) % 25 == 0:
        elapsed = time.time() - t0
        total_segs = sum(len(v) for v in ast_labels.values())
        print(f'  {vi+1}/{len(videos)} videos | {total_segs:,} segments | {elapsed/60:.1f}min', flush=True)

print(f'\nDONE: {len(ast_labels)} videos in {(time.time()-t0)/60:.1f}min')

In [ ]:
# Step 7: Analyze AST vs Energy agreement
all_ast = []
all_energy = []
all_vtt = []
for vid, labels in ast_labels.items():
    for l in labels:
        all_ast.append(l['ast_prob'])
        all_energy.append(l['rel_energy'])
        all_vtt.append(l['vtt_label'])

all_ast = np.array(all_ast)
all_energy = np.array(all_energy)
all_vtt = np.array(all_vtt)

ast_binary = (all_ast > 0.5).astype(int)
energy_binary = (all_energy > 2.0).astype(int)

print(f'Total segments: {len(all_ast):,}')
print(f'AST positive (>0.5): {ast_binary.sum():,} ({ast_binary.mean()*100:.1f}%)')
print(f'Energy positive (>2x): {energy_binary.sum():,} ({energy_binary.mean()*100:.1f}%)')
print(f'VTT positive: {all_vtt.sum():,} ({all_vtt.mean()*100:.1f}%)')

# Agreement
agree = (ast_binary == energy_binary).mean()
print(f'\nAST vs Energy agreement: {agree*100:.1f}%')

from sklearn.metrics import f1_score, confusion_matrix
print(f'\nConfusion Matrix (AST vs Energy):')
cm = confusion_matrix(energy_binary, ast_binary)
print(f'  Energy\\AST  Neg    Pos')
print(f'  Neg       {cm[0,0]:6d}  {cm[0,1]:6d}')
print(f'  Pos       {cm[1,0]:6d}  {cm[1,1]:6d}')

if ast_binary.sum() > 0 and all_vtt.sum() > 0:
    print(f'\nAST vs VTT F1: {f1_score(all_vtt, ast_binary):.4f}')
    print(f'Energy vs VTT F1: {f1_score(all_vtt, energy_binary):.4f}')

# Save results
with open('/content/gdrive/MyDrive/ast_validation_results.json', 'w') as f:
    json.dump({vid: labels for vid, labels in ast_labels.items()}, f)
print('\nSaved to Drive!')

## Part B: CNN on Mel Spectrograms

In [ ]:
# Step 8: Prepare mel spectrograms for CNN
def audio_to_mel(y, sr=SR, n_mels=80, hop=256):
    if len(y) < sr * 0.1:
        return np.zeros((1, n_mels, 87), dtype=np.float32)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, hop_length=hop, n_fft=512)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    if mel_db.shape[1] < 87:
        mel_db = np.pad(mel_db, ((0,0),(0, 87-mel_db.shape[1])))
    else:
        mel_db = mel_db[:, :87]
    return mel_db[np.newaxis].astype(np.float32)

# Sample 100 videos for CNN (full audio loading is expensive)
np.random.seed(42)
cnn_vids = sorted(overlap)
np.random.shuffle(cnn_vids)
cnn_vids = cnn_vids[:200]  # 200 videos

print(f'Loading mel spectrograms for {len(cnn_vids)} videos...')
train_X, train_y, test_X, test_y = [], [], [], []
split_idx = int(0.8 * len(cnn_vids))
t0 = time.time()

for vi, vid in enumerate(cnn_vids):
    audio_path = audio_map[vid]
    try:
        y_full, sr = librosa.load(audio_path, sr=SR, mono=True)
    except: continue
    
    vid_utts = utts_by_vid.get(vid, [])
    step = max(1, len(vid_utts) // 30)
    
    for u in vid_utts[::step][:30]:
        s, e = int(u['start']*SR), int(u['end']*SR)
        if e > len(y_full): e = len(y_full)
        seg = y_full[s:e]
        mel = audio_to_mel(seg)
        
        # Energy label
        full_rms = np.sqrt(np.mean(y_full**2)) + 1e-8
        seg_rms = np.sqrt(np.mean(seg**2)) if len(seg) > 0 else 0
        label = 1 if seg_rms / full_rms > 2.0 else 0
        
        if vi < split_idx:
            train_X.append(mel)
            train_y.append(label)
        else:
            test_X.append(mel)
            test_y.append(label)
    
    if (vi+1) % 25 == 0:
        print(f'  {vi+1}/{len(cnn_vids)} | Train:{len(train_y)} Test:{len(test_y)} | {(time.time()-t0)/60:.1f}min', flush=True)

train_X = np.array(train_X, dtype=np.float32)
train_y = np.array(train_y)
test_X = np.array(test_X, dtype=np.float32)
test_y = np.array(test_y)

# Normalize
mean, std = train_X.mean(), train_X.std()
train_X = (train_X - mean) / (std + 1e-8)
test_X = (test_X - mean) / (std + 1e-8)

print(f'\nTrain: {len(train_y)} ({train_y.mean()*100:.1f}% pos) | Test: {len(test_y)} ({test_y.mean()*100:.1f}% pos)')
print(f'Mel shape: {train_X.shape}')

In [ ]:
# Step 9: Train CNN on GPU
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class MelCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(32), nn.Dropout(0.2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(64), nn.Dropout(0.2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.BatchNorm2d(128), nn.Dropout(0.2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(-1)

class DS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

model = MelCNN().to(device)
pos_w = torch.tensor([(len(train_y)-train_y.sum())/max(train_y.sum(),1)]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

train_loader = DataLoader(DS(train_X, train_y), batch_size=128, shuffle=True)
test_loader = DataLoader(DS(test_X, test_y), batch_size=128)

best_f1 = 0
for epoch in range(50):
    model.train()
    for x, yb in train_loader:
        x, yb = x.to(device), yb.to(device)
        optimizer.zero_grad()
        criterion(model(x), yb).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    scheduler.step()
    
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, yb in test_loader:
            x = x.to(device)
            p = (torch.sigmoid(model(x)) > 0.5).cpu().numpy()
            preds.extend(p)
            labels.extend(yb.numpy())
    f1 = f1_score(labels, preds)
    if f1 > best_f1:
        best_f1 = f1
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}: F1 = {f1:.4f} (best: {best_f1:.4f})')

print(f'\nCNN BEST F1: {best_f1:.4f}')
print(f'(Energy LR on same videos: ~0.97)')

# Save model
torch.save(model.state_dict(), '/content/gdrive/MyDrive/cnn_mel_gpu.pt')
print('Saved to Drive!')

In [ ]:
# Step 10: Summary
print('='*60)
print('GPU VALIDATION COMPLETE')
print('='*60)
print(f'AST segments: {len(all_ast):,}')
print(f'AST vs Energy agreement: {agree*100:.1f}%')
print(f'CNN F1: {best_f1:.4f}')
print(f'Energy LR F1: ~0.97')
print()
print('If AST agrees with energy labels > 80%, labels are VALIDATED.')
print('If CNN F1 > 0.90, CNN learns same patterns from raw audio.')